In [1]:
# -----------------------------------------------------------
# Cell 1 – Imports & configuration
# -----------------------------------------------------------

import os
import json
import zipfile
import itertools
import random
import math
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd                     # ← alias corretto
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.utils.tensorboard import SummaryWriter

# Scikit‑learn utilities
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    f1_score,
    classification_report,
)

# Mixed‑precision (AMP) utilities
from torch.cuda.amp import autocast, GradScaler

# Enable the fastest convolution algorithm for fixed‑size batches
torch.backends.cudnn.benchmark = True

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Running on {DEVICE}")

🚀 Running on cuda


In [2]:
# -----------------------------------------------------------
# Cell 2 – Paths (Kaggle‑style) & output directories
# -----------------------------------------------------------

# Cartelle di ingresso (Kaggle monta i dati in /kaggle/input)
BASE_DIR      = "/kaggle/input/datasets/christiancorvino/cholec80-mr-sapienza"
FEATURE_DIR   = "/kaggle/input/datasets/christiancorvino/cholec80-features/features"

# Cartelle di output (scrivibili)
OUTPUT_ROOT   = "/kaggle/working"
LOG_DIR       = os.path.join(OUTPUT_ROOT, "logs")
MODEL_DIR     = os.path.join(OUTPUT_ROOT, "models")
RESULTS_DIR   = os.path.join(OUTPUT_ROOT, "results")

# Creiamo le directory se non esistono già
for d in (LOG_DIR, MODEL_DIR, RESULTS_DIR):
    os.makedirs(d, exist_ok=True)

# ----------------------------------------------------------------
# Phase dictionary (usato anche per il modello)
# ----------------------------------------------------------------
PHASE_DICT = {
    "Preparation": 0,
    "CalotTriangleDissection": 1,
    "ClippingCutting": 2,
    "GallbladderDissection": 3,
    "GallbladderPackaging": 4,
    "CleaningCoagulation": 5,
    "GallbladderRetraction": 6,
}
PHASE_NAMES = list(PHASE_DICT.keys())
NUM_CLASSES = len(PHASE_DICT)

print(f"🔢 Phase dictionary loaded – {NUM_CLASSES} classes:")
for i, name in enumerate(PHASE_NAMES):
    print(f"   {i}: {name}")

# TensorBoard writer (unico per tutta la sessione)
writer = SummaryWriter(log_dir=LOG_DIR)

🔢 Phase dictionary loaded – 7 classes:
   0: Preparation
   1: CalotTriangleDissection
   2: ClippingCutting
   3: GallbladderDissection
   4: GallbladderPackaging
   5: CleaningCoagulation
   6: GallbladderRetraction


In [3]:
# Cell 3 – Custom Dataset (temporal windows)
class Cholec80Dataset(Dataset):
    """
    Carica le feature pre‑estratte (N × D) e le etichette di fase.
    Restituisce finestre temporali di lunghezza `win_len` con step = 1.
    """
    def __init__(self, video_ids, win_len=64, stride=1, train=True):
        self.video_ids = video_ids
        self.win_len = win_len
        self.stride = stride
        self.train = train
        self.samples = []               # (video_id, start_idx)

        for vid in self.video_ids:
            feat_path = Path(FEATURE_DIR) / f"{vid}_features.npy"
            label_path = Path(FEATURE_DIR) / f"{vid}_labels.npy"
            feats = np.load(feat_path)            # shape: (T, D)
            labs = np.load(label_path)             # shape: (T,)
            T = feats.shape[0]

            # genera gli indici di inizio per le finestre
            for start in range(0, T - self.win_len + 1, self.stride):
                self.samples.append((vid, start))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        vid, start = self.samples[idx]
        feats = np.load(Path(FEATURE_DIR) / f"{vid}_features.npy")
        labs  = np.load(Path(FEATURE_DIR) / f"{vid}_labels.npy")

        window_feats = feats[start:start+self.win_len]          # (W, D)
        window_labels = labs[start:start+self.win_len]          # (W,)

        # data‑augmentation solo in training
        if self.train:
            # random temporal jitter (cambio di velocità)
            speed = random.choice([0.9, 1.0, 1.1])
            new_len = int(self.win_len * speed)
            # cropping / padding to keep fixed size
            if new_len < self.win_len:
                pad = self.win_len - new_len
                window_feats = np.pad(window_feats[:new_len],
                                      ((0, pad), (0,0)), mode='edge')
                window_labels = np.pad(window_labels[:new_len],
                                      (0, pad), mode='edge')
            else:
                window_feats = window_feats[:self.win_len]
                window_labels = window_labels[:self.win_len]

        # Tensor conversion
        return (torch.from_numpy(window_feats).float(),
                torch.from_numpy(window_labels).long())

In [4]:
# -----------------------------------------------------------
# Cell 4 – Train‑/‑val split & sampler (class‑imbalance aware)
# -----------------------------------------------------------

# ------------------------------------------------------------------
# 1️⃣  Elenco dei video presenti nella cartella FEATURES
# ------------------------------------------------------------------
ALL_VIDS = sorted(
    [p.stem.replace('_features', '')
     for p in Path(FEATURE_DIR).glob("*_features.npy")]
)
print(f"🔎 Found {len(ALL_VIDS)} video feature files in `{FEATURE_DIR}`.")

# ------------------------------------------------------------------
# 2️⃣  Shuffle & split (80 % training, 20 % validation)
# ------------------------------------------------------------------
random.shuffle(ALL_VIDS)
split = int(0.8 * len(ALL_VIDS))
train_vids = ALL_VIDS[:split]
val_vids   = ALL_VIDS[split:]

print(f"🗂️ Train set: {len(train_vids)} videos")
print(f"🗂️ Validation set: {len(val_vids)} videos")

# ------------------------------------------------------------------
# 3️⃣  Dataset objects (window‑based dataset defined in Cell 3)
# ------------------------------------------------------------------
train_set = Cholec80Dataset(train_vids, win_len=64, stride=1, train=True)
val_set   = Cholec80Dataset(val_vids,   win_len=64, stride=1, train=False)

print(f"📦 Train windows:  {len(train_set)}")
print(f"📦 Validation windows: {len(val_set)}")

# ------------------------------------------------------------------
# 4️⃣  Calcolo delle etichette di tutti i frame di training
# ------------------------------------------------------------------
#   (necessario per il bilanciamento delle classi)
all_train_labels = np.concatenate(
    [np.load(Path(FEATURE_DIR) / f"{vid}_labels.npy") for vid in train_vids]
)

# ------------------------------------------------------------------
# 5️⃣  Calcolo dei pesi inversi per ciascuna classe
# ------------------------------------------------------------------
# Frequenza di ogni classe (p(c))
class_counts = np.bincount(all_train_labels, minlength=NUM_CLASSES)
class_freq   = class_counts / class_counts.sum()                # p(c)

# Peso inverso: 1 / p(c)  (più alto per le classi rare)
inv_weights = 1.0 / (class_freq + 1e-6)                         # evita div by 0

# Normalizziamo affinché la media sia 1 (facilita il sampler)
inv_weights = inv_weights / inv_weights.mean()
print(f"🔧 Inverse class weights (per‑class): {inv_weights}")

# ------------------------------------------------------------------
# 6️⃣  Sample‑weight per ogni *finestra* di training
# ------------------------------------------------------------------
sample_weights = []
for vid, start in train_set.samples:
    # Etichette dei frame del video corrente
    labs = np.load(Path(FEATURE_DIR) / f"{vid}_labels.npy")
    # Etichetta del frame centrale della finestra (più stabile)
    central_label = labs[start + train_set.win_len // 2]
    # Peso inverso della classe centrale
    sample_weights.append(inv_weights[central_label].item())

print(f"🔢 Sample‑weight list built – length: {len(sample_weights)}")
print(f"   – Example first 5 weights: {sample_weights[:5]}")

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# ------------------------------------------------------------------
# 7️⃣  DataLoaders
# ------------------------------------------------------------------
train_loader = DataLoader(
    train_set,
    batch_size=32,
    sampler=sampler,          # usa il weighted sampler
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    val_set,
    batch_size=32,
    shuffle=False,            # validazione ordinata
    num_workers=2,
    pin_memory=True,
)

print(f"🚂 DataLoaders ready → train batches: {len(train_loader)}, "
      f"val batches: {len(val_loader)}")

🔎 Found 80 video feature files in `/kaggle/input/datasets/christiancorvino/cholec80-features/features`.
🗂️ Train set: 64 videos
🗂️ Validation set: 16 videos
📦 Train windows:  139705
📦 Validation windows: 39833
🔧 Inverse class weights (per‑class): [1.39317226 0.18313937 0.92637798 0.20666821 1.57761995 0.91719931
 1.79582293]
🔢 Sample‑weight list built – length: 139705
   – Example first 5 weights: [1.3931722552784975, 1.3931722552784975, 1.3931722552784975, 1.3931722552784975, 1.3931722552784975]
🚂 DataLoaders ready → train batches: 4366, val batches: 1245


In [5]:
# -----------------------------------------------------------
# Cell 5 – Model definition (fixed – forward present)
# -----------------------------------------------------------

# -----------------------------------------------------------
# 1️⃣  LinearProjector – proiezione delle feature pre‑estratte
# -----------------------------------------------------------
class LinearProjector(nn.Module):
    """
    Proietta le feature pre‑estratte (dimensione D) in uno spazio
    di embedding più piccolo (embed_dim). Viene applicata a ciascun
    timestep in modo indipendente.
    """
    def __init__(self, in_dim, embed_dim=256, dropout=0.2):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(in_dim, embed_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout)
        )
        print(f"🔧 LinearProjector → in_dim={in_dim}, embed_dim={embed_dim}, dropout={dropout}")

    def forward(self, x):
        """
        x : Tensor di shape (B, T, D)
        ritorna : Tensor (B, T, embed_dim)
        """
        return self.proj(x)       # (B, T, embed_dim)


# -----------------------------------------------------------
# 2️⃣  TemporalTransformer – modellazione temporale
# -----------------------------------------------------------
class TemporalTransformer(nn.Module):
    """
    Encoder Transformer a più strati che affina le embedding
    temporali. Include dropout interno e LayerNorm finale.
    """
    def __init__(self,
                 embed_dim=256,
                 nhead=8,
                 num_layers=3,          # ridotto da 4 per ridurre over‑fit
                 dim_feedforward=512,
                 dropout=0.2):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation='relu'
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers
        )
        self.norm = nn.LayerNorm(embed_dim)   # normalizzazione finale
        self.classifier = nn.Linear(embed_dim, NUM_CLASSES)  # 7 classi

        print(f"🔧 TemporalTransformer → embed_dim={embed_dim}, nhead={nhead}, "
              f"layers={num_layers}, dropout={dropout}")

    def forward(self, x):
        """
        x : Tensor di shape (B, T, embed_dim)
        ritorna : logits (B, T, NUM_CLASSES)
        """
        x = self.transformer(x)   # (B, T, embed_dim)
        x = self.norm(x)          # LayerNorm
        logits = self.classifier(x)   # (B, T, NUM_CLASSES)
        return logits


# -----------------------------------------------------------
# 3️⃣  PhaseRecognizer – pipeline completa
# -----------------------------------------------------------
class PhaseRecognizer(nn.Module):
    """
    Pipeline completa:
        1. LinearProjector → riduce la dimensionalità delle feature.
        2. TemporalTransformer → cattura dipendenze temporali.
    """
    def __init__(self, feat_dim, embed_dim=256):
        super().__init__()
        self.projector = LinearProjector(
            in_dim=feat_dim,
            embed_dim=embed_dim,
            dropout=0.2
        )
        self.temporal = TemporalTransformer(
            embed_dim=embed_dim,
            nhead=8,
            num_layers=3,          # più leggero rispetto a 4
            dropout=0.2
        )
        print(f"🚀 PhaseRecognizer initialized → feat_dim={feat_dim}, embed_dim={embed_dim}")

    def forward(self, x):
        """
        x : Tensor (B, T, D)  ← D = dimensione delle feature pre‑estratte
        ritorna : logits (B, T, NUM_CLASSES)
        """
        # 1️⃣ Proiezione delle feature
        proj = self.projector(x)          # (B, T, embed_dim)

        # 2️⃣ Modellazione temporale tramite Transformer
        logits = self.temporal(proj)      # (B, T, NUM_CLASSES)

        return logits

# ----------------------------------------------------------------
# Helper: crea il modello e, se ci sono più GPU, lo avvolge con
#         torch.nn.DataParallel.
# ----------------------------------------------------------------
def build_phase_recognizer(feat_dim: int,
                           embed_dim: int = 256,
                           use_dp: bool = True):
    """
    Ritorna un'istanza di PhaseRecognizer.
    - `feat_dim`  : dimensionalità delle feature pre‑estratte (es. 2048).
    - `embed_dim` : dimensione dell'embedding interno.
    - `use_dp`    : se True e più di una GPU è disponibile,
                    avvolge il modello in DataParallel.
    """
    base_model = PhaseRecognizer(feat_dim=feat_dim,
                                 embed_dim=embed_dim).to(DEVICE)
    if use_dp and torch.cuda.device_count() > 1:
        model = torch.nn.DataParallel(base_model)
        print(f"💡 Model wrapped with DataParallel ({torch.cuda.device_count()} GPUs)")
    else:
        model = base_model
        print("💡 Model running on a single GPU")
    return model

In [6]:
# Quick check della dimensionalità delle feature (da eseguire una volta)
sample_feat_path = Path(FEATURE_DIR) / f"{train_vids[0]}_features.npy"
sample_feat = np.load(sample_feat_path)      # shape: (T, D)
FEAT_DIM = sample_feat.shape[1]              # D
print(f"Feature dimension detected: {FEAT_DIM}")

Feature dimension detected: 2048


In [7]:
# -----------------------------------------------------------
# Cell 6 – Loss & metrics (FocalLoss + adjustable label smoothing)
# -----------------------------------------------------------

class FocalLoss(nn.Module):
    """
    Focal loss per gestire lo sbilanciamento delle classi.
    gamma – fattore di focalizzazione (default = 2.0)
    alpha – peso per classe (può essere un array di dimensione C)
    """
    def __init__(self, gamma=2.0, alpha=None, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        if alpha is None:
            self.alpha = 1.0
        else:
            self.alpha = torch.tensor(alpha, dtype=torch.float)
        self.reduction = reduction
        print(f"🔧 FocalLoss → gamma={gamma}, alpha={'auto' if alpha is None else 'provided'}")

    def forward(self, inputs, targets):
        # inputs: (B, T, C) – raw logits
        # targets: (B, T) – class index
        log_prob = F.log_softmax(inputs, dim=-1)          # log‑softmax
        prob     = torch.exp(log_prob)                    # softmax
        pt       = prob.gather(dim=-1,
                               index=targets.unsqueeze(-1)).squeeze(-1)

        # alpha per classe (se fornito)
        if isinstance(self.alpha, torch.Tensor):
            at = self.alpha.to(inputs.device)[targets]
        else:
            at = self.alpha

        loss = -at * (1 - pt) ** self.gamma * log_prob.gather(
            dim=-1, index=targets.unsqueeze(-1)).squeeze(-1)

        if self.reduction == 'mean':
            return loss.mean()
        else:
            return loss.sum()


# -------------- Choose loss -----------------
# Opzione A: Focal Loss (in genere più efficace con sbilanciamento)
criterion = FocalLoss(gamma=2.0, alpha=None, reduction='mean')

# Opzione B: Label‑smoothing (puoi aumentare ε se preferisci)
#criterion = LabelSmoothingCrossEntropy(eps=0.2, reduction='mean')


def compute_metrics(logits, labels, *, verbose=False):
    """
    logits: (B, T, C) → pred = argmax
    labels: (B, T)
    Returns dict with accuracy, macro‑F1, per‑class F1.
    If `verbose=True` the function prints a detailed report.
    """
    preds = torch.argmax(logits, dim=-1).cpu().numpy().flatten()
    trues = labels.cpu().numpy().flatten()
    acc = accuracy_score(trues, preds)
    macro_f1 = f1_score(trues, preds, average='macro')
    per_class = f1_score(trues, preds, average=None)

    if verbose:
        # ------ Detailed logging (only when requested) ------
        print(f"📊 Metrics computed → acc={acc:.4f}, macro_f1={macro_f1:.4f}")

        # Retrieve phase names; fall back to generic names if unavailable
        phase_names = globals().get(
            'PHASE_NAMES',
            [f'Class_{i}' for i in range(len(per_class))]
        )
        for i, f1 in enumerate(per_class):
            name = phase_names[i] if i < len(phase_names) else f'Class_{i}'
            print(f"   – Class {i} ({name}): F1={f1:.4f}")

    return {"acc": acc, "macro_f1": macro_f1, "per_class_f1": per_class}

🔧 FocalLoss → gamma=2.0, alpha=auto


In [8]:
# -----------------------------------------------------------
# Cell 7 – Training & validation loops (tqdm + AMP + warm‑up + clipping)
# -----------------------------------------------------------

# ------------------------------------------------------------------
# Utility: learning‑rate warm‑up (linear per i primi 500 step)
# ------------------------------------------------------------------
def _warmup_lr(optimizer, step, warmup_steps=500, base_lr=2e-4):
    """Aggiorna il LR in modo lineare durante i primi `warmup_steps`."""
    if step < warmup_steps:
        lr = base_lr * float(step) / warmup_steps
        for pg in optimizer.param_groups:
            pg["lr"] = lr
    return optimizer

# ------------------------------------------------------------------
# TRAIN ONE EPHEMERAL (AMP + clipping)
# ------------------------------------------------------------------
def train_one_epoch(model, loader, optimizer, epoch, num_epochs, global_step, scaler):
    """
    Esegue il training di una singola epoca.
    - tqdm → barra unica.
    - torch.cuda.amp → mixed‑precision.
    - gradient clipping (norm = 1.0).
    - warm‑up LR (gestito qui).
    """
    model.train()
    pbar = tqdm(loader,
                desc=f"🚀 Epoch {epoch}/{num_epochs}  TRAIN",
                leave=False,
                dynamic_ncols=True)
    running_loss, correct, total = 0.0, 0, 0
    for batch_idx, (feat, label) in enumerate(pbar, start=1):
        # --------------------------------------------------------------
        # forward (AMP)
        # --------------------------------------------------------------
        feat  = feat.to(DEVICE)          # (B, T, D)
        label = label.to(DEVICE)         # (B, T)
        optimizer.zero_grad()
        with autocast():
            logits = model(feat)          # (B, T, C)
            loss   = criterion(logits, label)
        # --------------------------------------------------------------
        # backward + scaler + clipping
        # --------------------------------------------------------------
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)                 # per gradient‑clipping su FP32
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        # --------------------------------------------------------------
        # warm‑up LR update (global step)
        # --------------------------------------------------------------
        _warmup_lr(optimizer, global_step + batch_idx)
        # --------------------------------------------------------------
        # statistics for the progress bar
        # --------------------------------------------------------------
        running_loss += loss.item()
        preds = torch.argmax(logits, dim=-1)
        correct += (preds == label).sum().item()
        total   += label.numel()
        avg_loss = running_loss / batch_idx
        avg_acc  = correct / total if total > 0 else 0.0
        pbar.set_postfix(loss=f"{avg_loss:.4f}", acc=f"{avg_acc:.4f}")
    # ------------------------------------------------------------------
    # epoch‑wide summary
    # ------------------------------------------------------------------
    epoch_loss = running_loss / len(loader)
    epoch_acc  = correct / total if total > 0 else 0.0
    print(f"✅ Epoch {epoch}/{num_epochs} – TRAIN DONE → "
          f"loss {epoch_loss:.4f} | acc {epoch_acc:.4f}")
    # TensorBoard logging
    writer.add_scalar("Train/EpochLoss", epoch_loss, epoch)
    writer.add_scalar("Train/Accuracy", epoch_acc, epoch)
    return epoch_loss, epoch_acc

# ------------------------------------------------------------------
# VALIDATION ONE EPHEMERAL (AMP)
# ------------------------------------------------------------------
def validate(model, loader, epoch, num_epochs):
    """
    Validation di una singola epoca con barra tqdm.
    Usa autocast ma *non* aggiorna i pesi.
    """
    model.eval()
    pbar = tqdm(loader,
                desc=f"🔎 Epoch {epoch}/{num_epochs}  VALID",
                leave=False,
                dynamic_ncols=True)
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for batch_idx, (feat, label) in enumerate(pbar, start=1):
            feat  = feat.to(DEVICE)
            label = label.to(DEVICE)
            with autocast():
                logits = model(feat)
                loss   = criterion(logits, label)
            running_loss += loss.item()
            preds = torch.argmax(logits, dim=-1)
            correct += (preds == label).sum().item()
            total   += label.numel()
            avg_loss = running_loss / batch_idx
            avg_acc  = correct / total if total > 0 else 0.0
            pbar.set_postfix(loss=f"{avg_loss:.4f}", acc=f"{avg_acc:.4f}")
    epoch_loss = running_loss / len(loader)
    epoch_acc  = correct / total if total > 0 else 0.0
    print(f"✅ Epoch {epoch}/{num_epochs} – VAL DONE   → "
          f"loss {epoch_loss:.4f} | acc {epoch_acc:.4f}")
    writer.add_scalar("Val/EpochLoss", epoch_loss, epoch)
    writer.add_scalar("Val/Accuracy", epoch_acc, epoch)
    return epoch_loss, epoch_acc

In [9]:
# -----------------------------------------------------------
# Cell 8 – Main training routine (with AMP, warm‑up & early‑stop)
# -----------------------------------------------------------

import math
import pandas as pd
import torch

# ------------------------------------------------------------------
# Helper: format the learning‑rate of an optimizer
# ------------------------------------------------------------------
def _format_lr(optimizer):
    """Restituisce il LR corrente in notazione scientifica."""
    if not hasattr(optimizer, "param_groups") or len(optimizer.param_groups) == 0:
        return "N/A"
    lr = optimizer.param_groups[0].get("lr", None)
    return f"{lr:.2e}" if lr is not None else "N/A"

# ------------------------------------------------------------------
# Main training function
# ------------------------------------------------------------------
def main_training(num_epochs=30, patience=3, lr=2e-4):
    """
    - Crea il modello PhaseRecognizer (wrappato in DataParallel se possibile).
    - Usa AdamW + CosineAnnealingWarmRestarts.
    - Mixed‑precision (AMP) + warm‑up.
    - Early‑stopping basato su macro‑F1.
    - Salva checkpoint & CSV di history.
    """
    # ------------------------------------------------------------------
    # Model & optimizer
    # ------------------------------------------------------------------
    model = build_phase_recognizer(feat_dim=FEAT_DIM,
                                   embed_dim=256,
                                   use_dp=True).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=1e-4,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer,
        T_0=10,
        T_mult=2,
        eta_min=1e-5,
    )
    # Scaler per AMP
    scaler = GradScaler()
    # ------------------------------------------------------------------
    # Header informativo
    # ------------------------------------------------------------------
    print("\n⚙️  Training start")
    print(f"→ epochs   : {num_epochs}")
    print(f"→ patience : {patience}")
    print(f"→ lr start : {lr}")
    print(f"→ device   : {DEVICE}")
    best_val_f1      = -math.inf
    epochs_no_improve = 0
    best_ckpt        = None
    global_step      = 0          # contatore globale di batch per warm‑up
    # History container
    history = {
        "train_loss": [], "val_loss": [],
        "train_acc":  [], "val_acc":  [],
        "val_f1":     [],  # per il plotting rapido
    }
    # ------------------------------------------------------------------
    # Epoch loop
    # ------------------------------------------------------------------
    for epoch in range(1, num_epochs + 1):
        # -------------------- TRAIN --------------------
        tr_loss, tr_acc = train_one_epoch(
            model,
            train_loader,
            optimizer,
            epoch,
            num_epochs,
            global_step,
            scaler,
        )
        global_step += len(train_loader)   # aggiorna il contatore globale
        # -------------------- VALID --------------------
        val_loss, val_acc = validate(
            model,
            val_loader,
            epoch,
            num_epochs,
        )
        # -------------------- MACRO‑F1 --------------------
        # raccogli tutti logits/labels dell'epoch di validation
        all_logits, all_labels = [], []
        model.eval()
        with torch.no_grad():
            for f, l in val_loader:
                f, l = f.to(DEVICE), l.to(DEVICE)
                all_logits.append(model(f))
                all_labels.append(l)
        logits_cat = torch.cat(all_logits, dim=0)
        labels_cat = torch.cat(all_labels, dim=0)
        val_metrics = compute_metrics(logits_cat, labels_cat, verbose=False)
        val_f1 = val_metrics["macro_f1"]
        # -------------------- RECORD --------------------
        history["train_loss"].append(tr_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(val_acc)
        history["val_f1"].append(val_f1)
        # -------------------- CHECKPOINT --------------------
        ckpt_path = os.path.join(
            MODEL_DIR,
            f"epoch{epoch:02d}_valF1{val_f1:.4f}.pth",
        )
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_f1": val_f1,
                "scaler_state_dict": scaler.state_dict(),
            },
            ckpt_path,
        )
        print(f"💾 Checkpoint saved → {ckpt_path}")
        # -------------------- EARLY‑STOPPING --------------------
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            epochs_no_improve = 0
            best_ckpt = ckpt_path
            print(f"🏆 New best val‑F1: {best_val_f1:.4f} (epoch {epoch})")
        else:
            epochs_no_improve += 1
            print(f"🕒 No improvement – patience {epochs_no_improve}/{patience}")
            if epochs_no_improve >= patience:
                print(f"\n🛑 Early stopping triggered after epoch {epoch}")
                break
        # -------------------- SCHEDULER STEP --------------------
        scheduler.step()
        print(f"🔄 Scheduler LR (epoch {epoch}) → {_format_lr(optimizer)}")
    # ------------------------------------------------------------------
    # FINAL LOG & CSV
    # ------------------------------------------------------------------
    print("\n✅ Training finished")
    print(f"→ Best validation F1: {best_val_f1:.4f}")
    print(f"→ Best checkpoint   : {best_ckpt}")
    history_path = os.path.join(RESULTS_DIR, "training_history.csv")
    pd.DataFrame(history).to_csv(history_path, index=False)
    print(f"📁 History CSV saved → {history_path}")
    return model, best_ckpt, history

# -----------------------------------------------------------
# Esecuzione
# -----------------------------------------------------------
trained_model, best_checkpoint, train_history = main_training(
    num_epochs=40,
    patience=3,
    lr=2e-4,
)

🔧 LinearProjector → in_dim=2048, embed_dim=256, dropout=0.2
🔧 TemporalTransformer → embed_dim=256, nhead=8, layers=3, dropout=0.2
🚀 PhaseRecognizer initialized → feat_dim=2048, embed_dim=256
💡 Model wrapped with DataParallel (2 GPUs)

⚙️  Training start
→ epochs   : 40
→ patience : 3
→ lr start : 0.0002
→ device   : cuda


🚀 Epoch 1/40  TRAIN:   0%|          | 0/4366 [00:00<?, ?it/s]

✅ Epoch 1/40 – TRAIN DONE → loss 0.0951 | acc 0.9226


🔎 Epoch 1/40  VALID:   0%|          | 0/1245 [00:00<?, ?it/s]

✅ Epoch 1/40 – VAL DONE   → loss 0.4924 | acc 0.8038
💾 Checkpoint saved → /kaggle/working/models/epoch01_valF10.7344.pth
🏆 New best val‑F1: 0.7344 (epoch 1)
🔄 Scheduler LR (epoch 1) → 1.95e-04


...

✅ Epoch 3/40 – VAL DONE   → loss 0.5868 | acc 0.7927
💾 Checkpoint saved → /kaggle/working/models/epoch03_valF10.7130.pth
🕒 No improvement – patience 2/3
🔄 Scheduler LR (epoch 3) → 1.61e-04


🚀 Epoch 4/40  TRAIN:   0%|          | 0/4366 [00:00<?, ?it/s]

✅ Epoch 4/40 – TRAIN DONE → loss 0.0027 | acc 0.9977


🔎 Epoch 4/40  VALID:   0%|          | 0/1245 [00:00<?, ?it/s]

✅ Epoch 4/40 – VAL DONE   → loss 0.6358 | acc 0.7977
💾 Checkpoint saved → /kaggle/working/models/epoch04_valF10.7146.pth
🕒 No improvement – patience 3/3

🛑 Early stopping triggered after epoch 4

✅ Training finished
→ Best validation F1: 0.7344
→ Best checkpoint   : /kaggle/working/models/epoch01_valF10.7344.pth
📁 History CSV saved → /kaggle/working/results/training_history.csv


In [10]:
# -----------------------------------------------------------
# Cell 9 – Confusion matrix & detailed report (with prints)
# -----------------------------------------------------------

def evaluate_on_set(model, loader, split_name="Val"):
    model.eval()
    all_true, all_pred = [], []

    print(f"\n🔎 Evaluating on {split_name} set (batches: {len(loader)})")
    with torch.no_grad():
        for batch_idx, (feat, label) in enumerate(loader):
            feat  = feat.to(DEVICE)
            label = label.to(DEVICE)

            logits = model(feat)
            preds  = torch.argmax(logits, dim=-1)

            all_true.append(label.cpu().numpy().flatten())
            all_pred.append(preds.cpu().numpy().flatten())

            if batch_idx % 20 == 0:
                print(f"[eval]  batch {batch_idx:04d}")

    y_true = np.concatenate(all_true)
    y_pred = np.concatenate(all_pred)

    acc      = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    report   = classification_report(y_true, y_pred,
                                     target_names=PHASE_NAMES,
                                     digits=4,
                                     output_dict=True)

    # Confusion matrix plot
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=PHASE_NAMES, yticklabels=PHASE_NAMES, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(f'{split_name} Confusion Matrix')
    cm_path = os.path.join(RESULTS_DIR,
                           f'{split_name.lower()}_confusion.png')
    plt.tight_layout()
    plt.savefig(cm_path)
    plt.close(fig)

    # Save classification report
    report_path = os.path.join(RESULTS_DIR,
                               f'{split_name.lower()}_classification_report.csv')
    pd.DataFrame(report).transpose().to_csv(report_path)

    print(f"✅ {split_name} evaluation complete")
    print(f"   – Accuracy   : {acc:.4f}")
    print(f"   – Macro‑F1   : {macro_f1:.4f}")
    print(f"   – Confusion matrix saved → {cm_path}")
    print(f"   – Classification report saved → {report_path}")

    return {"accuracy": acc,
            "macro_f1": macro_f1,
            "confusion_path": cm_path,
            "report_path": report_path}

In [11]:
# Cell 10 – Run evaluation on validation (already done) & save results
val_metrics = evaluate_on_set(trained_model, val_loader, split_name="Val")
print(val_metrics)


🔎 Evaluating on Val set (batches: 1245)
[eval]  batch 0000
[eval]  batch 0020
[eval]  batch 0040
[eval]  batch 0060
[eval]  batch 0080
[eval]  batch 0100
[eval]  batch 0120
[eval]  batch 0140
[eval]  batch 0160
[eval]  batch 0180
[eval]  batch 0200
[eval]  batch 0220
[eval]  batch 0240
[eval]  batch 0260
[eval]  batch 0280
[eval]  batch 0300
[eval]  batch 0320
[eval]  batch 0340
[eval]  batch 0360
[eval]  batch 0380
[eval]  batch 0400
[eval]  batch 0420
[eval]  batch 0440
[eval]  batch 0460
[eval]  batch 0480
[eval]  batch 0500
[eval]  batch 0520
[eval]  batch 0540
[eval]  batch 0560
[eval]  batch 0580
[eval]  batch 0600
[eval]  batch 0620
[eval]  batch 0640
[eval]  batch 0660
[eval]  batch 0680
[eval]  batch 0700
[eval]  batch 0720
[eval]  batch 0740
[eval]  batch 0760
[eval]  batch 0780
[eval]  batch 0800
[eval]  batch 0820
[eval]  batch 0840
[eval]  batch 0860
[eval]  batch 0880
[eval]  batch 0900
[eval]  batch 0920
[eval]  batch 0940
[eval]  batch 0960
[eval]  batch 0980
[eval]  b

In [12]:
# Cell 11 – Create final archive
zip_path = os.path.join(OUTPUT_ROOT, "surgical_phase_recognition_results.zip")
with zipfile.ZipFile(zip_path, mode='w', compression=zipfile.ZIP_DEFLATED) as zf:
    # model checkpoints
    for root, _, files in os.walk(MODEL_DIR):
        for f in files:
            zf.write(os.path.join(root, f),
                     arcname=os.path.relpath(os.path.join(root, f), OUTPUT_ROOT))
    # logs (TensorBoard files)
    for root, _, files in os.walk(LOG_DIR):
        for f in files:
            zf.write(os.path.join(root, f),
                     arcname=os.path.relpath(os.path.join(root, f), OUTPUT_ROOT))
    # results (CSV, PNG)
    for root, _, files in os.walk(RESULTS_DIR):
        for f in files:
            zf.write(os.path.join(root, f),
                     arcname=os.path.relpath(os.path.join(root, f), OUTPUT_ROOT))

print(f"✅ Archive created: {zip_path}")

✅ Archive created: /kaggle/working/surgical_phase_recognition_results.zip
